In [1]:
import torch 
import lightning as L
import yaml
import sys, os
sys.path.append('../')
sys.path.append('./')
import importlib
from pathlib import Path 
# from byol_a.models import AudioNTT2020



In [2]:
sys.path.append("lightning_scripts")
import lightning_scripts.eval_jsin_transfer_matched as lightning 
from byol_a.common import *

importlib.reload(lightning)
BYOLAClassifier = lightning.BYOLAClassifier

config_path = 'byol-a/config.yaml'
config = load_yaml_config(config_path)

config['model'] = {}
config['hparas'] = {}
config['hparas']['task_loss_params'] = {
    "signal/word_int":
      {"loss_type": 'crossentropyloss',
      "weight": 1.0},                       # init loss is ~6.6 
   "noise/labels_int":
      {"loss_type": 'bcewithlogitsloss',
      "weight": 1.0},                      # init loss is ~200 
    "signal/speaker_int":
      {"loss_type": 'crossentropyloss',
      "weight": 1.0}
    }

config['audio_transforms'] = {} 
config['audio_transforms']['low_snr'] = -10
config['audio_transforms']['high_snr'] = 10
config['audio_transforms']['rms_level'] = 60

config['model']['arch_kwargs'] = {}
config['data'] = {}
config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794,    
                                    "signal/speaker_int": 433} 
task_str = f"word_and_speaker_task"
config['hparas']['batch_size'] = 32
config['hparas']['lr'] = 0.01
config['data']['eval_max'] = 3
config['hparas']['optimizer'] = "AdamW"
config['hparas']['epochs'] = 1
config['num_workers'] = 1 
config['with_noise'] = True

config['hparas']['task_loss_params'] = {key:value for key,value in config['hparas']['task_loss_params'].items() if key in config['model']['arch_kwargs']['num_classes'].keys()}

config['data']['target_keys'] = list(config['model']['arch_kwargs']['num_classes'].keys())

# task_config = {}
# task_config['num_classes'] = 784



module = BYOLAClassifier(config)

/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/common.py:31: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")
/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/models.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals

In [3]:
config_path

'byol-a/config.yaml'

In [4]:
classifier_ckpt_path = 'model_checkpoints/config/linear_classifier_checkpoints_word_and_speaker_task_final_full_rep_AdamW_0.01_cosine_lr_scheduler__w_dropout/epoch=5-step=181000.ckpt'
classifier_ckpt = torch.load(classifier_ckpt_path, weights_only=False) # get latest checkpoint 
module.load_state_dict(classifier_ckpt['state_dict'])

<All keys matched successfully>

In [5]:
from jsinV3DataLoader_precombined_batched import jsinV3_precombined_all_signals, CleanSpeechInNoiseValDatasetBatched


def eval_collate_fn(batch):
    audio, targets = batch[0] # unbox wrapper added by dataloader 
    audio = audio.unsqueeze(1)
    # # combine labels: each target is dict for each key, stack the values 
    labels = {}
    for label_key in targets.keys():
        labels[label_key] = torch.from_numpy(targets[label_key])
    return audio, labels



eval_speech_h5_path = '/mnt/home/jfeather/ceph/data/training_datasets_a' \
'udio/jsinV3BalancedProcessed/sr_20000/splits/train_stackedDataframeHDF_n150_VJRUH4IEPDGPNH2JZMULSQKOWYNQ6KMM.pdh5'

test_dataset = CleanSpeechInNoiseValDatasetBatched(speech_h5_path=eval_speech_h5_path,
                                        target_keys=config['data']['target_keys'],
                                        batch_size=config['hparas']['batch_size'],
                                        )

test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    num_workers=config['num_workers'],
    shuffle=False,
    collate_fn=eval_collate_fn
)

In [14]:
trainer = L.Trainer(
    precision="32",
    # limit_val_batches=0,
    max_epochs=config['hparas']['epochs'],
    devices=1,
    accelerator="gpu", 
    # val_check_interval = 2000, 
    # limit_train_batches=2,
    limit_predict_batches=100,
    gradient_clip_val=1, # clipt grad l2 norm to 1 
    profiler=None,
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [16]:
outputs = trainer.predict(module, test_dataloader, return_predictions=True)
outputs

top1_word = []
top1_speaker = []
top5_word = []
top5_speaker = []

for record in outputs:
    top1_word.append(record['top1']['signal/word_int'])
    top5_word.append(record['top5']['signal/word_int'])
    top1_speaker.append(record['top1']['signal/speaker_int'])
    top5_speaker.append(record['top5']['signal/speaker_int'])
n_examples = len(outputs)

output_dict = {
    "word_top1_mean": torch.stack(top1_word).mean(),
    "word_top1_sem": torch.stack(top1_word).std() / np.sqrt(n_examples),
    "speaker_top1_mean": torch.stack(top1_speaker).mean(),
    "speaker_top1_sem": torch.stack(top1_speaker).std() / np.sqrt(n_examples),

    "word_top5_mean": torch.stack(top5_word).mean(),
    "word_top5_sem": torch.stack(top5_word).std() / np.sqrt(n_examples),
    "speaker_top5_mean": torch.stack(top5_speaker).mean(),
    "speaker_top5_sem": torch.stack(top5_speaker).std() / np.sqrt(n_examples),
}
output_dict

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

tensor([635, 635, 603, 635, 635, 635, 635, 635, 548, 271, 219, 635, 635, 635,
        635, 635, 573, 635, 635, 322, 635, 185, 635, 635, 715, 635, 271, 635,
        322, 548, 603, 271], device='cuda:0')
(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31],
       device='cuda:0'),)
tensor([635, 635, 603, 635, 635, 635, 635, 635, 548, 271, 219, 635, 635, 635,
        635, 635, 573, 635, 635, 322, 635, 185, 635, 635, 715, 635, 271, 635,
        322, 548, 603, 271], device='cuda:0')
tensor([293, 211, 260, 164, 260, 211, 211, 242, 211, 260, 211, 211, 164, 211,
        425, 425, 164, 211, 164, 293, 425, 242, 242, 242, 293, 211, 242, 164,
        421, 242, 242, 164], device='cuda:0')
(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31],
       device='cuda:0'),)
tensor([293, 211, 260, 164, 260, 211, 211, 242, 21

{'word_top1_mean': tensor(0.0034),
 'word_top1_sem': tensor(0.0010),
 'speaker_top1_mean': tensor(0.0622),
 'speaker_top1_sem': tensor(0.0039),
 'word_top5_mean': tensor(0.2400),
 'word_top5_sem': tensor(0.0331),
 'speaker_top5_mean': tensor(0.3003),
 'speaker_top5_sem': tensor(0.0171)}